> **SUPERSEDED PROTOCOL ARTIFACT.** This notebook uses task-level `personas_n3`. Use `contribution_insights_hardneg_v1.json` and `.md` for the corrected primary.\n\n## tl;dr

- The main contribution is not merely a four-point accuracy curve: report-level persona recoverability changes state across stages.
- Search is the bottleneck: 27 reports go from correct at Planning to wrong at Search, versus only 4 moving in the opposite direction.
- Recovery is concrete rather than only aggregate: 21/27 Planning-to-Search losses are attributable again at Writing.
- Solar's advantage over BM25 is smallest at Search (+0.100; 95% task-bootstrap CI [0.008, 0.192]) and largest at Writing (+0.217 [0.100, 0.333]) across both seeds.
- Corrected two-seed identifier masking changes stage accuracy by at most 1.7 points, with every interval including zero.


## Context & Methods

This companion notebook selects defensible manuscript contributions from the final PDR-Bench artifacts. The primary population is 120 reports from 20 task clusters and two generation seeds. Baseline and corrected identifier-masking comparisons both use all 120 reports. Task-cluster bootstrap intervals use 5,000 deterministic resamples.

### Key Assumptions

- Acc@1 measures recoverability of the conditioning persona, not report quality or causal influence.
- Solar-vs-baseline comparisons are paired by report and stage.
- Seed comparisons pair the same task/persona across seeds, but only two seeds are available.


## Data

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "manifest.json").exists()),
    None,
)
if root is None:
    raise RuntimeError("Could not locate the repository root")

subprocess.run(
    [sys.executable, "scripts/analyze_contribution_insights.py"],
    cwd=root,
    check=True,
)
insights = json.loads(
    (root / "paper/analysis/contribution_insights.json").read_text()
)
quality = pd.read_csv(
    root / "runs/confirmatory/analysis_sha256/quality_sensitivity.csv"
)
search_views = json.loads(
    (root / "runs/confirmatory/analysis_search_views/search_view_summary.json").read_text()
)
insights["population"], search_views["population"], quality.shape


Contribution analysis → /Users/janghyeonseo/Desktop/DRA/paper/analysis/contribution_insights.json


({'baseline_reports': 120,
  'masking_reports': 120,
  'primary_reports': 120,
  'primary_tasks': 20,
  'seeds': [0, 1]},
 {'full': 120, 'queries': 120, 'snippets': 120},
 (16, 5))

## Results

### 1. Report-level state transitions

In [2]:
flows = pd.DataFrame(insights["adjacent_stage_flows"])
flows[[
    "from_stage", "to_stage", "correct_to_correct",
    "correct_to_wrong", "wrong_to_correct", "wrong_to_wrong", "net_gain"
]]


,from_stage,to_stage,correct_to_correct,correct_to_wrong,wrong_to_correct,wrong_to_wrong,net_gain
0,plan,search,61,27,4,28,-23
1,search,compress,56,9,17,38,8
2,compress,write,67,6,18,29,12


In [3]:
pd.Series(insights["recovery"], name="reports").to_frame()


,reports
of_planning_losses_recovered_by_compression,13
of_planning_losses_recovered_by_writing,21
planning_correct_search_wrong,27
search_wrong,55
search_wrong_but_compression_correct,17
search_wrong_but_writing_correct,29


### 2. Baselines and Search artifact views

In [4]:
baseline = pd.DataFrame(insights["solar_vs_baselines"])
baseline = baseline[baseline["scope"] == "all"]
display(baseline[[
    "baseline", "stage", "solar_minus_baseline", "ci95_low", "ci95_high",
    "n_reports"
]])
search_comparisons = pd.DataFrame(search_views["comparisons"])
search_comparisons[search_comparisons["scope"] == "all"][[
    "left", "right", "left_accuracy", "right_accuracy", "left_minus_right",
    "ci95_low", "ci95_high", "n_reports"
]]


,baseline,stage,solar_minus_baseline,ci95_low,ci95_high,n_reports
8,bm25,plan,0.150000,0.075000,0.225000,120
9,bm25,search,0.100000,0.008333,0.191667,120
10,bm25,compress,0.108333,0.025000,0.191667,120
11,bm25,write,0.216667,0.100000,0.333333,120
20,embedding,plan,0.266667,0.141667,0.383333,120
21,embedding,search,0.183333,0.050000,0.316667,120
22,embedding,compress,0.241667,0.158333,0.333333,120
23,embedding,write,0.225000,0.108333,0.350000,120
32,random,plan,0.491667,0.316667,0.675000,120
33,random,search,0.208333,0.083333,0.341667,120


,left,right,left_accuracy,right_accuracy,left_minus_right,ci95_low,ci95_high,n_reports
2,queries,full,0.608333,0.541667,0.066667,0.016667,0.116667,120
5,snippets,full,0.525000,0.541667,-0.016667,-0.100000,0.083333,120
8,queries,snippets,0.608333,0.525000,0.083333,0.000000,0.166667,120


### 3. Seed stability, masking, and quality sensitivity

In [5]:
seed_comparison = pd.DataFrame(insights["seed1_vs_seed0"])
masking = pd.DataFrame(insights["identifier_masked_minus_original"])
display(seed_comparison[["stage", "seed1_minus_seed0", "ci95_low", "ci95_high"]])
display(masking[["stage", "masked_minus_original", "ci95_low", "ci95_high"]])
quality.pivot(index="stage", columns="policy", values="accuracy")


,stage,seed1_minus_seed0,ci95_low,ci95_high
0,plan,0.000000,-0.083333,0.083333
1,search,-0.050000,-0.183333,0.083333
2,compress,-0.016667,-0.166667,0.133333
3,write,-0.116667,-0.233333,-0.033333


,stage,masked_minus_original,ci95_low,ci95_high
0,plan,-0.008333,-0.033333,0.016667
1,search,0.008333,-0.041667,0.066667
2,compress,0.000000,-0.041667,0.041667
3,write,0.016667,-0.016667,0.050000


policy,all_matched,no_completeness_errors,no_ledger_errors,success_criteria_met
stage,,,,
compress,0.608333,0.648352,0.605042,0.644444
plan,0.733333,0.769231,0.731092,0.766667
search,0.541667,0.571429,0.537815,0.566667
write,0.708333,0.758242,0.705882,0.755556


## Takeaways

1. **Stage-wise attribution is a transition diagnostic.** The 27-to-4 asymmetry at Planning→Search and later 17-to-9 and 18-to-6 recovery flows reveal where individual reports lose and regain recoverability.
2. **The Search bottleneck has two components.** Query-only accuracy is 0.608, 6.7 points above the full query-plus-snippet view, while snippet-only is 0.525 and indistinguishable from full; nearly every query is already visible in full.
3. **The contextual-matcher advantage is smallest at Search.** Across both seeds, Solar is 10.0 points above BM25 at Search and 21.7 points above it at Writing; both task-bootstrap intervals are above zero, while the gap magnitudes vary by seed.
4. **The recovery is not driven by named entities or failed artifacts.** Corrected two-seed masking changes no stage by more than 1.7 points, and both quality-filtered subsets retain the trajectory.
5. **Writing is recoverable but generation-sensitive.** Seed 1 is 11.7 points below seed 0 at Writing; the task-bootstrap interval excludes zero. The paper should claim a cross-seed dip-and-recovery shape, not seed-invariant endpoint magnitude.
6. **Claim boundary.** These findings support a diagnostic protocol and a recoverability result, not a utility or causal-personalization claim.
